In [9]:
%cd /content/Master_Thesis
!apt-get install tree -y
!tree /content/Master_Thesis
!ls "master thesis"


/content/Master_Thesis
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tree is already the newest version (2.0.2-1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
/content/Master_Thesis
├── DistilBERT.ipynb
├── emotion_intensity_data
│   ├── dev.csv
│   ├── test.csv
│   └── train.csv
├── RoBERTa_Hierarchical.ipynb
├── RoBERTa.ipynb
├── RoBERTa_Log.csv
├── RoBERTa_SingleStep.ipynb
├── test2.ipynb
└── test.ipynb

1 directory, 10 files
ls: cannot access 'master thesis': No such file or directory


In [10]:
!git add .
!git commit -m "update csv log"
!git push



Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@3d6695f253dc.(none)')
To https://github.com/sharayumhaske22/Master_Thesis.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/sharayumhaske22/Master_Thesis.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [1]:
import numpy as np
import torch
import time
import os
import csv

from datasets import load_dataset, concatenate_datasets
from scipy.stats import pearsonr

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    TrainerCallback,
)


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="train")
val= load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="dev")
test= load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="test")
print(train)  # Print the first example to understand its structure

train_df = train.to_pandas()
print("Sample data (first 5 rows):")
print(train_df.head())

full_data = concatenate_datasets([train, val, test])

# Total size
len_total = len(full_data)

# Exact 70/20/10 counts
target_train= int(round(0.70 * len_total))
target_val   = int(round(0.20 * len_total))
target_test  = len_total - target_train - target_val

split_1 = full_data.train_test_split(train_size=target_train, seed=42)
train_data = split_1["train"]
remaining  = split_1["test"]

# Split remaining into val and test
split_2 = remaining.train_test_split(train_size=target_val, seed=42)
val_data  = split_2["train"]
test_data = split_2["test"]

print("Train size:", len(train_data))
print("Validation size:", len(val_data))
print("Test size:", len(test_data))



Device: cuda
Dataset({
    features: ['id', 'text', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise'],
    num_rows: 2763
})
Sample data (first 5 rows):
                        id                                               text  \
0  eng_train_track_b_00001                       Colorado, middle of nowhere.   
1  eng_train_track_b_00002  This involved swimming a pretty large lake tha...   
2  eng_train_track_b_00003        It was one of my most shameful experiences.   
3  eng_train_track_b_00004  After all, I had vegetables coming out my ears...   
4  eng_train_track_b_00005                        Then the screaming started.   

   anger  disgust  fear  joy  sadness  surprise  
0      0      NaN     1    0        0         1  
1      0      NaN     2    0        0         0  
2      0      NaN     1    0        3         0  
3      0      NaN     0    0        0         0  
4      0      NaN     3    0        1         2  
Train size: 3950
Validation size: 1129
Test size: 56

In [3]:
TEXT_COL = "text"

EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]

In [4]:
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess(example):
    encoded = tokenizer(
        example[TEXT_COL],
        truncation=True,
        max_length=256,
    )
    encoded["labels"] = [float(example[e]) for e in EMOTIONS]
    return encoded

train_tok = train_data.map(preprocess)
val_tok   = val_data.map(preprocess)
test_tok  = test_data.map(preprocess)

cols = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch", columns=cols)
test_tok.set_format(type="torch", columns=cols)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1129 [00:00<?, ? examples/s]

Map:   0%|          | 0/564 [00:00<?, ? examples/s]

In [5]:
cols = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch", columns=cols)
test_tok.set_format(type="torch", columns=cols)

In [6]:
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(EMOTIONS),
    problem_type="regression"
).to(device)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
def safe_pearson(x, y):
    r, _ = pearsonr(x, y)
    return 0.0 if np.isnan(r) else float(r)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.asarray(preds)
    labels = np.asarray(labels)

    metrics = {}
    rs = []

    for i, emo in enumerate(EMOTIONS):
        r = safe_pearson(preds[:, i], labels[:, i])
        metrics[f"pearson_{emo}"] = r
        rs.append(r)

    metrics["pearson_mean"] = float(np.mean(rs))
    return metrics

In [16]:
import csv
import os

LOG_FILE = "RoBERTa_Log.csv"

print("Working directory:", os.getcwd())
print("Absolute path:", os.path.abspath(LOG_FILE))

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "eval_loss", "accuracy"])
    f.flush()  # force write

# Immediately read the file after writing
with open(LOG_FILE, "r", encoding="utf-8") as f:
    content = f.read()

print("File content after writing:")
print(content)

Working directory: /content
Absolute path: /content/RoBERTa_Log.csv
File content after writing:
epoch,train_loss,eval_loss,accuracy



In [8]:
LOG_FILE = "RoBERTa_Log.csv"

if os.path.exists(LOG_FILE):
    print(f"CSV file exists at: {LOG_FILE}")
else:
    print("CSV file not found. It will be created during training.")

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "eval_loss", "accuracy"])  # accuracy = pearson_mean

class SaveMetricsCallback(TrainerCallback):
    def __init__(self, file_path):
        self.file_path = file_path
        self.last_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.last_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            epoch = metrics.get("epoch", state.epoch)
            eval_loss = metrics.get("eval_loss", "")
            accuracy = metrics.get("eval_pearson_mean", "")

            with open(self.file_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([epoch, self.last_train_loss, eval_loss, accuracy])


CSV file not found. It will be created during training.


In [10]:
EPOCHS = 5


training_args = TrainingArguments(
    output_dir="roberta_brighter_onlyintensities",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=EPOCHS,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="pearson_mean",
    greater_is_better=True,

    report_to="none",
    fp16=torch.cuda.is_available(),
)

# ==============================
# 12) Trainer
# ==============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[SaveMetricsCallback(LOG_FILE)],
)

# ==============================
# 13) Train
# ==============================
start = time.time()
trainer.train()
end = time.time()

print(f"\nTotal training time: {end - start:.1f} seconds")
print(f"Average time per epoch: {(end - start) / EPOCHS:.1f} seconds")




Epoch,Training Loss,Validation Loss,Pearson Anger,Pearson Fear,Pearson Joy,Pearson Sadness,Pearson Surprise,Pearson Mean
1,0.594553,0.589723,-0.002773,0.048455,0.039527,0.023479,0.046162,0.030970
2,0.586176,0.589267,0.021270,0.092947,0.038354,0.033857,0.212181,0.079722
3,0.574217,0.580557,0.026676,0.123061,0.025462,0.050824,0.312011,0.107607
4,0.566193,0.567029,0.027683,0.135990,0.028622,0.062557,0.321363,0.115243
5,0.560154,0.567472,0.026546,0.150567,0.025657,0.068260,0.321262,0.118458


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Total training time: 470.0 seconds
Average time per epoch: 94.0 seconds


In [ ]:
print("CSV log saved as:", LOG_FILE)

CSV log saved as: roberta_loss_accuracy.csv


In [ ]:
def predict_intensities(text: str):
    model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits.detach().cpu().numpy()[0]

    discrete = np.clip(np.rint(logits), 0, 3).astype(int)

    return {
        EMOTIONS[i]: {"raw": float(logits[i]), "intensity_0_3": int(discrete[i])}
        for i in range(len(EMOTIONS))
    }

print(predict_intensities("I feel so happy and joyful today!"))

{'anger': {'raw': 0.345703125, 'intensity_0_3': 0}, 'fear': {'raw': 0.625, 'intensity_0_3': 1}, 'joy': {'raw': 0.8740234375, 'intensity_0_3': 1}, 'sadness': {'raw': 0.25048828125, 'intensity_0_3': 0}, 'surprise': {'raw': 0.88818359375, 'intensity_0_3': 1}}
